# Barter Python: Trading Statistics and Performance Analysis

This notebook converts the Rust statistics walkthrough into the current Python bindings for `Instrument`, `IndexedInstruments`, `TradingSummaryGenerator`, and `TradingSummary`.

## Prerequisites

These notebooks assume `barter` is installed into the active IPython kernel, for example:

```bash
cd /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python
uv sync --dev
source .venv/bin/activate
python -m pip install -e .
```

If you register a dedicated kernel, select that kernel before running the notebook.


In [ ]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        python_dir = candidate / "barter-python" / "python" / "barter"
        if (python_dir / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate the barter-rs repository root from the current working directory.")

REPO_ROOT = find_repo_root()
PYTHON_SOURCE = REPO_ROOT / "barter-python" / "python"
if str(PYTHON_SOURCE) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE))

print(f"Using barter from {PYTHON_SOURCE}")


In [ ]:
import json
from datetime import datetime, timedelta, timezone

from barter import ExchangeId, IndexedInstruments, Instrument, TradingSummaryGenerator


## 1. Define Instruments

`IndexedInstruments` is the Python entry point for the same indexed lookup model used in Rust.

In [ ]:
btc = Instrument.spot(ExchangeId.BINANCE_SPOT, "btc_usdt", "BTCUSDT", "btc", "usdt")
eth = Instrument.spot(ExchangeId.BINANCE_SPOT, "eth_usdt", "ETHUSDT", "eth", "usdt")
instruments = IndexedInstruments([btc, eth])

print(instruments)
print(instruments.instruments())

## 2. Feed Synthetic Balance and Position Updates

The Python generator exposes `update_balance()` and `update_position()` so you can build summaries without constructing the full Rust engine state by hand.

In [ ]:
generator = TradingSummaryGenerator(instruments, 0.05)

start = datetime(2025, 1, 1, tzinfo=timezone.utc)
asset_index_usdt = 2

balance_points = [10000.0, 10400.0, 10150.0, 10900.0]
for day, total in enumerate(balance_points):
    generator.update_balance(
        asset_index=asset_index_usdt,
        total=total,
        free=total,
        time_exchange=(start + timedelta(days=day)).isoformat().replace("+00:00", "Z"),
    )

positions = [
    (0, "buy", 400.0, 0.05, 0, 2),
    (0, "buy", -250.0, 0.04, 3, 5),
    (1, "buy", 650.0, 0.75, 6, 8),
]

for instrument_index, side, pnl, qty, enter_day, exit_day in positions:
    generator.update_position(
        instrument_index=instrument_index,
        side=side,
        pnl_realised=pnl,
        quantity=qty,
        time_enter=(start + timedelta(days=enter_day)).isoformat().replace("+00:00", "Z"),
        time_exit=(start + timedelta(days=exit_day)).isoformat().replace("+00:00", "Z"),
    )

print(generator)

## 3. Generate a Summary

Use `daily` for day-based metrics or `annual365` for 24/7 crypto-style annualisation.

In [ ]:
summary = generator.generate("annual365")
summary_dict = summary.to_dict()

print(summary)
print(sorted(summary_dict.keys()))
print(json.dumps(summary_dict, indent=2)[:4000])